In [1]:
import os
from elasticsearch import Elasticsearch, helpers, NotFoundError
import json
from datetime import datetime

In [2]:
USER="elastic"
PWD="M4mz6mUi"

In [3]:
client = Elasticsearch("http://localhost:9200", \
                        basic_auth=(USER, PWD))

In [4]:
client.info()

ObjectApiResponse({'name': 'aad12e08edcd', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'TGcuRaidT8-NFRNPvtnPPQ', 'version': {'number': '8.17.3', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'a091390de485bd4b127884f7e565c0cad59b10d2', 'build_date': '2025-02-28T10:07:26.089129809Z', 'build_snapshot': False, 'lucene_version': '9.12.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [5]:
#!mkdir answers

In [6]:
#q1
resp = client.cluster.health()
first_five_pairs = dict(list(resp.items())[:5])
with open('answers/q1.json', 'w') as f:
    json.dump(dict(first_five_pairs), f, indent=4)

In [7]:
#q2
data_dir = 'data/jsons'
madmap = "madmap"

try:
    client.indices.delete(index=madmap)
except NotFoundError:
    print("Index doesn't exist!")

client.indices.create(index=madmap)

json_files = [f for f in os.listdir(data_dir) if f.endswith('.json') and f != 'places.json']
operations = []
for file_name in json_files:
    with open(os.path.join(data_dir, file_name), 'r') as file:
        data = json.load(file)
        if "arrests" in data:
            records = data["arrests"]
        elif "articles" in data:
            records = data["articles"]
        elif type(data) is list:
            records = data
        else:
            records = [data]
        for record in records:
            operations.append({"_index": madmap, "_source": record})

helpers.bulk(client, operations)
mapping = dict(client.indices.get_mapping(index=madmap))
with open('answers/q2.json', 'w') as f:
    json.dump(mapping, f, indent=4)

In [8]:
#q3
txt_data_dir = "data/text"
txt_files = [f for f in os.listdir(txt_data_dir) if ".txt" in f]
txt_files
documents = []
for txt in txt_files:
    with open(os.path.join(txt_data_dir, txt), "r") as f:
        data = f.read()
        document = {"wiki": str(data)}
        documents.append(document)

helpers.bulk(client, documents, index=madmap)
mapping = dict(client.indices.get_mapping(index=madmap))
with open('answers/q3.json', 'w') as f:
    json.dump(mapping, f, indent=4)

In [9]:
# This is some helper code that I needed to get some questions below to work, primarily questions 4 and 7. 
# I gave chatGPT my code for both parts 4 and 7, as well as autograder feedback for both of those questions. 
# The main thing the autograder was saying was that there was no results, which didn't make sense to us because
# we were able to run '!head' and see that there is more than 0 results. 

# This helper code that chatGPT provided to remedy this helps because here we aren't able to directly use json.load().
# This is because json.load() is intended for single structured json whereas our data has top-level json objects.
# The following code is improvised from what was provided by chatGPT to help to resolve this. Furthermore, images 
# we attached in our submission that contain the information that chatGPT provided us.

places_path = os.path.join("data", "jsons", "places.json")

# This file is in "bulk" NDJSON format: each doc is preceded
# by a line with the metadata. We read 2 lines at a time:
with open(places_path, "r") as f:
    lines = f.read().strip().splitlines()

operations = []
for i in range(0, len(lines), 2):
    meta_line = lines[i]       # e.g. {"index": {"_index":"places","_id":"199"}}
    doc_line = lines[i+1]      # e.g. {"formattedAddress":"...","name":"...","coordinates":...}

    # Parse both lines as JSON
    meta_json = json.loads(meta_line)
    doc_json = json.loads(doc_line)

    # Use _id from the metadata line
    _id = meta_json["index"]["_id"]

    # We'll index these documents into "madmap" (ignoring the original _index)
    action = {
        "_op_type": "index",
        "_index": "madmap",
        "_id": _id,
        "_source": doc_json
    }
    operations.append(action)

helpers.bulk(client, operations)

(307, [])

In [10]:
#q4
client.indices.refresh(index=madmap)
q4_resp = client.search(
    index="madmap",
    query={
        "match": {
            "formattedAddress": "University"
        }
    },
    size=10000
)
q4_mapping = dict(q4_resp)
with open('answers/q4.json', 'w') as f:
    json.dump(q4_mapping, f, indent=4)

In [11]:
#q5
client.indices.refresh(index=madmap)
q5_resp = client.search(
    index="madmap",
    _source=["title"],
    query={
        "match": {
            "title": {
                "query": "Madson",
                "fuzziness": "AUTO"
            }
        }
    },
    size=10000
)
q5_mapping = dict(q5_resp)
with open('answers/q5.json', 'w') as f:
    json.dump(q5_mapping, f, indent=4)

In [12]:
#q6
client.indices.refresh(index=madmap)
q6_resp = client.search(
    index="madmap",
    query={
        "bool": {
            "should": [
                {"match_phrase": {"title": "Wisconsin Badgers"}},
                {"match_phrase": {"description": "Wisconsin Badgers"}},
                {"match_phrase": {"content": "Wisconsin Badgers"}}
            ],
            "minimum_should_match": 1
        }
    },
    size=10000
)
q6_mapping = dict(q6_resp)
with open('answers/q6.json', 'w') as f:
    json.dump(q6_mapping, f, indent=4)

In [13]:
#q7
client.indices.refresh(index=madmap)
q7_resp = client.search(
    index="madmap",
    _source=["name", "formattedAddress"],
    query={
        "bool": {
            "must": [
                {"exists": {"field": "formattedAddress"}}
            ],
            "must_not": [
                {"match": {"formattedAddress": "Madison"}}
            ]
        }
    },
    size=10000
)
q7_mapping = dict(q7_resp)
with open('answers/q7.json', 'w') as f:
    json.dump(q7_mapping, f, indent=4)

In [14]:
#q8
client.indices.refresh(index=madmap)
q8_resp = client.search(
    index="madmap",
    _source=["wiki"],
    query={
        "simple_query_string": {
            "query": "rivalry^10 football^5 badgers",
            "fields": ["wiki"]
        }
    },
    size=10000
)
q8_mapping = dict(q8_resp)
with open('answers/q8.json', 'w') as f:
    json.dump(q8_mapping, f, indent=4)

In [15]:
#q9
client.indices.refresh(index=madmap)
q9_resp = client.search(
    index="madmap",
    _source=["wiki"],
    query={
        "match_phrase": {
            "wiki": "rivalry"
        }
    },
    highlight={
        "fields": {
            "wiki": {}
        }
    },
    size=1
)
q9_mapping = dict(q9_resp)
hits = q9_mapping.get("hits", {}).get("hits", [])

if hits:
    top_hit = hits[0]
    highlight = top_hit.get("highlight", {}).get("wiki", [])
    q9_res = {
        "wiki": highlight
    }
else:
    q9_res = {"wiki": []}

with open('answers/q9.json', 'w') as f:
    json.dump(q9_res, f, indent=4)

In [16]:
#q10
client.indices.refresh(index=madmap)
q10_resp = client.search(
    index="madmap",
    _source=["title", "source.name", "publishedAt"],
    query={
        "term": {
            "source.name.keyword": {
                "value": "Nasa"
            }
        }
    },
    size=10000
)
q10_mapping = dict(q10_resp)
with open('answers/q10.json', 'w') as f:
    json.dump(q10_mapping, f, indent=4)

In [17]:
#q11
client.indices.refresh(index=madmap)
q11_resp = client.search(
    index="madmap",
    query={
        "range": {
            "year": {
                "gte": 2001,
                "lte": 2019
            }
        }
    },
    aggs={
        "arrests_sum": {
            "sum": {
                "field": "arrests"
            }
        }
    },
    size=0
)
q11_mapping = dict(q11_resp)
arrests_sum = q11_mapping.get("aggregations", {}).get("arrests_sum", {}).get("value", 0)
with open('answers/q11.json', 'w') as f:
    json.dump(arrests_sum, f, indent=4)

In [18]:
#q12
client.indices.refresh(index=madmap)
q12_resp = client.search(
    index="madmap",
    aggs={
        "top_sources": {
            "terms": {
                "field": "source.name.keyword",
                "size": 10
            }
        }
    },
    size=0
)
q12_mapping = dict(q12_resp)
buckets = q12_mapping.get("aggregations", {}).get("top_sources", {}).get("buckets", [])
with open('answers/q12.json', 'w') as f:
    json.dump(buckets, f, indent=4)

In [19]:
#q13
client.indices.refresh(index=madmap)
q13_resp = client.search(
    index="madmap",
    query={
        "exists": {
            "field": "name"
        }
    },
    size=0
)
q13_mapping = dict(q13_resp)
count = q13_mapping.get("hits", {}).get("total", {}).get("value", 0)
with open('answers/q13.json', 'w') as f:
    json.dump(count, f, indent=4)

In [20]:
#q14
client.indices.refresh(index=madmap)
q14_resp = client.search(
    index="madmap",
    aggs={
        "unique_authors": {
            "cardinality": {
                "field": "author.keyword"
            }
        }
    },
    size=0
)
q14_mapping = dict(q14_resp)
authors_count = q14_mapping.get("aggregations", {}).get("unique_authors", {}).get("value", 0)
with open('answers/q14.json', 'w') as f:
    json.dump(authors_count, f, indent=4)

In [21]:
#q15
client.indices.refresh(index=madmap)
q15_resp = client.search(
    index="madmap",
    query={
        "range": {
            "year": {
                "gte": 2001,
                "lte": 2019
            }
        }
    },
    aggs={
        "attended_avg": {
            "avg": {
                "field": "attended"
            }
        }
    },
    size=0
)
q15_mapping = dict(q15_resp)
avg_attend = q15_mapping.get("aggregations", {}).get("attended_avg", {}).get("value", 0)
with open('answers/q15.json', 'w') as f:
    json.dump(avg_attend, f, indent=4)

In [22]:
!pytest autograder.py

============================= test session starts ==============================
platform linux -- Python 3.10.12, pytest-8.3.4, pluggy-1.5.0
rootdir: /home/ekamghotra/project-3-sam-and-ekam-1
plugins: anyio-4.8.0
collected 22 items                                                             

......................                                     [100%]

============================== 22 passed in 0.33s ==============================
